# Multi-version by member

Snapshot selection happens before chunk planning. Missing or invalid records stop the run. Use a separate RUN_ID for every member/range and coordinate arXiv request scheduling.


:::{note}
This is a generated, nonexecuting mirror of the operational notebook. The Book build never crawls arXiv or mounts Drive.

[Open this version in Colab](https://colab.research.google.com/github/hcmus-arxiv-nhom1/CodeRepo/blob/15e82ac09c30c577fce82cdf500bd8a6d02d7eb0/src/notebooks/arxiv_nhom1_multi_version_benchmark.ipynb) · [Canonical source](https://github.com/hcmus-arxiv-nhom1/CodeRepo/blob/15e82ac09c30c577fce82cdf500bd8a6d02d7eb0/src/notebooks/arxiv_nhom1_multi_version_benchmark.ipynb) · Download notebook: {download}`./arxiv_nhom1_multi_version_benchmark.ipynb`

CodeRepo notebook revision: `15e82ac09c30c577fce82cdf500bd8a6d02d7eb0`. The setup cell separately pins the implementation revision used by the crawler.
:::

In [ ]:
# Directory configuration: verify these paths for the mounted Drive.
from pathlib import Path

SNAPSHOT_PATH = "/content/arxiv-assigned-multi-version.jsonl"
SNAPSHOT_DRIVE_PATH = "/content/drive/MyDrive/arxiv-inputs/arxiv-assigned-multi-version.jsonl"
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1/multi-version"

# S3 manifest, bundles, and preparation logs.


In [ ]:
# Edit this cell. Use a NEW run ID for each new experiment.
MODE = "partition"  # sample, partition (partial assigned range), or final
RUN_ID = "multi-dang-0000-4999"  # Unique per member/range
MEMBER = "dang"  # Choose: dang, thien, tien, hieu
if MEMBER not in ("dang", "thien", "tien", "hieu"):
    raise ValueError("Invalid member")
RANGES = (MEMBER,)
OFFSET = 0  # Offset within the selected member range
COUNT = None  # All IDs in each selected range before snapshot filtering
CHUNK_SIZE = 100  # Number of selected IDs per chunk before phase partitioning
WITH_REFERENCES = True
SNAPSHOT_SHA256 = "4d457f0e78bea5bae7addad0e56f55e056949b61037925fd0412ef20b0e5a081"
CHECKPOINT_S = 600  # Checkpoint target in seconds; not a strict deadline
RESUME = False  # True in a new runtime after a saved run/timeout
TEAM_ID = "arxiv-nhom1"
INITIALIZE_TEAM_FOLDER = False  # Create DRIVE_DIR first; True only once to create TEAM.json
CODE_REF = "main"  # New runs resolve main to an exact SHA; resume uses the saved SHA
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Copy the filtered snapshot from Drive

Upload `arxiv-assigned-multi-version.jsonl` beside the full snapshot in your Drive input folder. Default: `MyDrive/arxiv-inputs/arxiv-assigned-multi-version.jsonl`; edit SNAPSHOT_DRIVE_PATH if your folder differs. This file contains only the multi-version group for the assigned IDs, retaining original records.

Run this cell to mount Drive and copy the filtered file to /content. The SHA-256 is checked before publishing the local file; a matching existing local file is reused. The Drive original is never modified or deleted. Crawl outputs and checkpoints are saved on Drive. No Kaggle download or Kaggle token is needed.

Use a fresh runtime so the full snapshot is absent from local disk. Copy/hash is setup time outside the benchmark; the filtered input is counted in benchmark disk measurements. For resume keep the same filtered bytes, path and run configuration.


In [ ]:
import hashlib
import json
import shutil
import tempfile
from pathlib import Path


def verify_snapshot(path):
    if not path.is_file():
        raise FileNotFoundError(f"Upload the filtered snapshot to: {path}")
    with path.open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    if actual != SNAPSHOT_SHA256:
        raise ValueError(f"Snapshot SHA-256 mismatch: {path}; check the input file")


if SNAPSHOT_PATH:
    snapshot_target = Path(SNAPSHOT_PATH)
    if snapshot_target.exists():
        verify_snapshot(snapshot_target)
        print("Reusing verified local snapshot:", snapshot_target)
    else:
        from google.colab import drive

        drive.mount("/content/drive")
        snapshot_source = Path(SNAPSHOT_DRIVE_PATH)
        if not snapshot_source.is_file():
            raise FileNotFoundError(
                f"Upload arxiv-assigned-multi-version.jsonl to {snapshot_source}; "
                "or edit SNAPSHOT_DRIVE_PATH to match its location."
            )
        snapshot_target.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(
            prefix="arxiv-input-", dir=snapshot_target.parent
        ) as temporary:
            copied = Path(temporary) / "snapshot.jsonl"
            shutil.copyfile(snapshot_source, copied)
            verify_snapshot(copied)
            copied.replace(snapshot_target)
        print("Copied and verified filtered snapshot:", snapshot_target)
    print("Snapshot bytes:", snapshot_target.stat().st_size)
else:
    print("Snapshot disabled; discovery will use the network fallback.")

## Authorize Google Drive

Use the Google account with access to the team folder. Drive may already be mounted for the input snapshot.


In [ ]:
import os
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import userdata

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
if not DRIVE.is_dir():
    raise FileNotFoundError("Create/check the intended team Drive folder before running")


## Resolve saved revision

For resume, read the saved Drive manifest and keep the same run ID and configuration.


In [ ]:
revision = CODE_REF
if RESUME:
    manifest_path = DRIVE / "runs" / RUN_ID / "MANIFEST.json"
    saved = json.loads(manifest_path.read_text())
    revision = saved["code_revision"]


## Check out the implementation

The crawler is pinned to the immutable revision shown below. Restart the runtime before changing revisions.


In [ ]:
repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)

## Install dependencies

Package installation is setup and is outside the benchmark session.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True
)
if not Path("/usr/bin/time").is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "time"], check=True)

## Optional API credentials and local workspace

ARXIV_CONTACT_EMAIL and S2_API_KEY are read from Colab Secrets if present. No values are printed.


In [ ]:
for name in ("ARXIV_CONTACT_EMAIL", "S2_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value.strip()
    print(name, "configured" if value else "not configured")
LOCAL_ROOT = Path("/content/m1-local") / uuid.uuid4().hex
if SNAPSHOT_PATH and not Path(SNAPSHOT_PATH).is_file():
    raise FileNotFoundError("Copy the filtered snapshot to SNAPSHOT_PATH before benchmarking")

## Preflight and exact scope

Review the snapshot-selected paper and version counts, revision, CPU runtime and destination before running. OFFSET and COUNT apply to each original member range before snapshot grouping. `main` is resolved to a commit SHA for a new run; resume uses the SHA saved in its manifest.


In [ ]:
from dataclasses import asdict

import psutil

from src.benchmark.checkpoints import prepare_drive
from src.benchmark.runner import BenchmarkConfig, plan_chunks, run_benchmark

prepare_drive(DRIVE, TEAM_ID, initialize=INITIALIZE_TEAM_FOLDER)
cfg = BenchmarkConfig(
    run_id=RUN_ID,
    mode=MODE,
    ranges=tuple(RANGES),
    offset=OFFSET,
    count=COUNT,
    chunk_size=CHUNK_SIZE,
    references=WITH_REFERENCES,
    snapshot=SNAPSHOT_PATH,
    snapshot_group="multi_version",
    storage_backend="drive",
    checkpoint_s=CHECKPOINT_S,
)
chunks = plan_chunks(cfg)

## Review scope and runtime

Confirm the CPU runtime, selected IDs and destination before starting. Use a fresh RUN_ID for a new experiment.


In [ ]:
print(asdict(cfg))
print("Selected IDs:", sum(len(c.ids) for c in chunks), "chunks:", len(chunks))
print("CPU count:", psutil.cpu_count(), "RAM MB:", psutil.virtual_memory().total / 1e6)
print("Storage: Drive", DRIVE)
subprocess.run(["lscpu"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)
# Summarize the selected IDs using the same verified snapshot.
from collections import Counter
selected_owner = {pid: chunk.owner for chunk in chunks for pid in chunk.ids}
by_owner = {owner: Counter() for owner in RANGES}
with Path(SNAPSHOT_PATH).open(encoding="utf-8") as snapshot_file:
    for line in snapshot_file:
        record = json.loads(line)
        owner = selected_owner.get(record.get("id"))
        if owner is not None:
            by_owner[owner]["papers"] += 1
            by_owner[owner]["versions"] += len(record["versions"])
for owner, counts in by_owner.items():
    print(f"{owner}: {counts['papers']} papers, {counts['versions']} versions")
print("Progress bars track terminal source states (including no_tex/unavailable); references are separate.")


## Crawl configuration

Multi-version papers are retrieved using version-specific arXiv requests. Coordinate arXiv access across group runtimes.


In [ ]:
RUN_CRAWL = False  # Enable only after reviewing selection and destination.


In [ ]:
print("Multi-version source: arXiv; no S3 bundle preparation required")


## Crawl and benchmark

Only the snapshot multi-version group is included in this run. References and source completion are measured separately.
Separate paper and version progress bars cover the current chunk, including restored terminal results.


In [ ]:
if RUN_CRAWL:
    result = run_benchmark(cfg, drive_root=DRIVE, local_root=LOCAL_ROOT, revision=REVISION, team_id=TEAM_ID, resume=RESUME)
    print(json.dumps(result, indent=2))
else:
    print("Crawl skipped: RUN_CRAWL=False")


## Inspect completion and measurements

Corpus coverage and timing completeness are separate. Session RSS includes the notebook/runner baseline and descendants; memory is sampled, so peaks can be missed. Session sampler wall includes crawl, retries and final corpus sync, but excludes initial manifest hash, input setup and final evidence publication. Discovery includes index preparation; do not add overlapping checkpoint or reference times to wall. Mean wall per required ID differs from per-paper lifecycle latency. Preserve logs and measurements even if finalization fails.


In [ ]:
import pandas as pd
from IPython.display import display

local = LOCAL_ROOT / RUN_ID
for name in ("coverage.json", "timing_audit.json"):
    p = local / "work" / name
    if p.exists():
        print(name, p.read_text())
audit = local / "book_csv/measurement_audit.json"
if audit.exists():
    print("Measurement audit:", audit.read_text())
errors = local / "work/cleanup_errors.json"
if errors.exists():
    print("Cleanup diagnostics:", errors.read_text())
for name in ("benchmark_summary.csv", "corpus_summary.csv"):
    p = local / "book_csv" / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
p = local / "book_csv/paper_attempt_timings.csv"
if p.exists():
    print("Per-paper attempts (first 10; all rows in ZIP)")
    display(pd.read_csv(p).head(10))

## Export and download benchmark evidence

The evidence ZIP contains reports and logs. Raw source files remain in storage.


In [ ]:
from google.colab import files

from src.benchmark.runner import export_evidence, export_rescue

local = LOCAL_ROOT / RUN_ID
try:
    archive = export_evidence(cfg, local)
except Exception as exc:
    print("Statistics export failed; creating raw local rescue:", exc)
    archive = export_rescue(local)
print("Evidence:", archive)
files.download(str(archive))

## Rescue local data after an error

If benchmark/finalization fails, run this cell before resetting the runtime. It packages local data, logs and measurements without contacting the output storage.


In [ ]:
from google.colab import files as rescue_files

from src.benchmark.runner import export_rescue as make_rescue

archive = make_rescue(LOCAL_ROOT / RUN_ID)
print("Raw local rescue:", archive)
rescue_files.download(str(archive))

## Resume in a fresh runtime

Keep RUN_ID, storage backend, chunk size, snapshot bytes/path and settings unchanged; set RESUME=True. Setup loads the saved implementation revision. A run created with an older revision resumes that revision; it does not acquire newer fixes. Confirm the prior runtime has stopped before resuming the same RUN_ID. SHIFT is no longer used.

Drive: the index describes the last committed generation. Changed files retain SHA-256 backups so restore can recover that generation if the plain mirror was updated before index publication. Existing runs that were already corrupted cannot be repaired automatically. The raw Drive mirror may be partially updated after an interruption; restore and verify before M2 consumption.

